# Machine learning prediction of subsurface temperature and geothermal screening

This notebook runs the local analysis of the BGS UK Digital Geothermal Catalogue v2. It models Great Britain equilibrium observations and treats measured heat flow as separate screening context. The target is measured temperature at reported depth, not recoverable energy or engineering feasibility.

Run from the project root. The workflow is implemented in `src/analyze.py`; it writes cleaned tables to `data/processed/`, and metrics and figures to `outputs/`. Download the BGS data into `data/raw/` first (see `data/README.md`).

## Research question and scope

Can reported subsurface temperature be predicted from depth, location and site context when evaluation withholds 50 km spatial blocks? Great Britain is the modelling extent because its coordinates use EPSG:27700. Cornwall is a coverage-limited case study, not a separate training set.

The downloaded archive contains GB and NI temperature, conductivity and heat-flow tables, site metadata, lineage codes and the BGS guide. The sheets contain an embedded units row and use `-9999` and `Not Entered` sentinels; the workflow removes/normalizes them before modelling.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
run = subprocess.run([sys.executable, str(ROOT / 'src' / 'analyze.py')], cwd=ROOT, check=True, capture_output=True, text=True)
print(run.stdout)
summary = json.loads((ROOT / 'outputs/metrics/analysis_summary.json').read_text(encoding='utf-8'))
metrics = pd.read_csv(ROOT / 'outputs/metrics/model_metrics.csv')
display(metrics)
summary

## Method and outputs

The workflow cleans GB temperature, conductivity, heat-flow and site metadata tables. It selects three BGS equilibrium categories for the target and compares a median baseline, a depth-only linear gradient baseline, random forest and histogram gradient boosting with seven numeric predictors. Missing predictors are median-imputed using training data only. A deterministic grouped split holds out complete 50 km BNG blocks.

Metrics are MAE and RMSE in °C and R². The notebook generates five figures: observation footprint, temperature versus depth, spatial holdout diagnostics, model MAE, and independent heat-flow observations. Figures are saved to `outputs/figures/`.

## Results snapshot (latest local run)

Cleaned temperature observations: 8,900 across 1,731 sites. Equilibrium subset: 5,367 observations across 108 sites. Spatial holdout: 771 observations from 19 sites in 12 blocks. Histogram gradient boosting: MAE 2.64 °C, RMSE 3.89 °C, R² 0.945. Linear gradient baseline (depth only, ~35 °C/km): MAE 3.70 °C, R² 0.910. Median baseline: MAE 10.91 °C. See `outputs/metrics/model_metrics.csv` for the full model comparison and `outputs/metrics/spatial_holdout_predictions.csv` for row-level predictions.

The provisional Cornwall rectangle contains 13 observations from 6 sites. It is not an official Cornwall boundary, so this count only signals sparse coverage and does not support a Cornwall-only model.

## Interpretation limits

Catalogue records are historically uneven and vary in measurement method, provenance and depth datum. A spatial holdout reduces leakage between nearby locations but does not remove catalogue selection bias or guarantee fully independent geographic validation. Cornwall’s provisional rectangle is not an official boundary. Maps show measured locations and must not be read as continuous resource surfaces. This analysis cannot establish capacity, drilling feasibility or local site suitability.

Source: [BGS UK Digital Geothermal Catalogue v2](https://doi.org/10.5285/9c324699-abf1-4412-ab9f-39308507082d). Contains data supplied by UKRI.